<a href="https://colab.research.google.com/github/Nebius-academy/LLM-Engineering-Essentials/blob/main/topic2/2.1_structured_inputs_and_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# LLM Engineering Essentials от Nebius Academy
Курс на github: [ссылка](https://github.com/Nebius-Academy/LLM-Engineering-Essentials/tree/main)
Автор: Алексей Умнов
Ссылки:
- [LinkedIn](www.linkedin.com/in/alex-umnov)
- Профиль в Discord: *alexumnov*, лучше всего отметить #nebius-academy.
Курс сейчас находится в разработке, скоро появятся дополнительные материалы. [Подпишитесь, чтобы быть в курсе](https://academy.nebius.com/llm-engineering-essentials/update/)
№ 2.1. Структурированные входы и выходы
В теме 1 мы научились подсказывать LLM так, чтобы он понимал, что вы от него хотите, и давал релевантный ответ. В этом блокноте мы продолжим обсуждение, понимая
* Как сделать промпты многоразовыми с помощью **шаблонов промптов**.
* Как гарантировать, что LLM создает свои результаты в удобном, легко анализируемом формате.
Давайте начнем с запуска кода, который поможет нам во всем блокноте:

In [ ]:
!pip install openai -qU

In [ ]:
from google.colab import userdata
from openai import OpenAI
import os

os.environ['NEBIUS_API_KEY'] = userdata.get("nebius_api_key")

nebius_client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

llama_model = "meta-llama/Llama-3.3-70B-Instruct"

def prettify_string(text, max_line_length=80):
    """Prints a string with line breaks at spaces to prevent horizontal scrolling.

    Args:
        text: The string to print.
        max_line_length: The maximum length of each line.
    """

    output_lines = []
    lines = text.split("\n")
    for line in lines:
        current_line = ""
        words = line.split()
        for word in words:
            if len(current_line) + len(word) + 1 <= max_line_length:
                current_line += word + " "
            else:
                output_lines.append(current_line.strip())
                current_line = word + " "
        output_lines.append(current_line.strip())  # Append the last line
    return "\n".join(output_lines)

def answer_with_llm(prompt: str,
                    system_prompt="You are a helpful assistant",
                    max_tokens=512,
                    client=nebius_client,
                    model=llama_model,
                    prettify=True,
                    temperature=None) -> str:

    messages = []

    if system_prompt:
        messages.append(
            {
                "role": "system",
                "content": system_prompt
            }
        )

    messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )

    if prettify:
        return prettify_string(completion.choices[0].message.content)
    else:
        return completion.choices[0].message.content


# Шаблоны промптов

В системе на основе LLM всегда существует уровень логики промптов, скрытый от пользователя. Например, ChatGPT, Claude, Gemini и другие имеют довольно сложные **системные промпты**, которые устанавливают правила и ограничения общения LLM с пользователем.
Однако в некоторых случаях системные промпты не являются достаточно гибким механизмом. Представьте себе, например,
* бот службы поддержки клиентов, которому необходимо знать географию пользователя, чтобы давать релевантные ответы о продуктах, доступных на местном уровне.
* бот поддержки железнодорожных служб, который должен быть в курсе сегодняшних забастовок железнодорожников и других бедствий.
Вероятно, вам придется вставить эту информацию в середину приглашения; и для таких вещей отличным инструментом являются **шаблоны промптов**.

По сути, **шаблон приглашения** — это строка шаблона, например
```python
"some fixed information {template placeholder 1}
some more fixed information {template placeholder 2}"
```
где заполнители шаблона должны быть заполнены непосредственно перед фактическим вызовом LLM.
Давайте проверим несколько изящных способов упаковки этой логики.
Прежде всего, вы можете написать свою собственную обертку. В приведенном ниже примере `m['content'].format(**kwargs)` позволяет поместить в сообщение пользователя столько форматирования, сколько вы пожелаете.

In [ ]:
from typing import List, Dict

class MessagesPromptTemplate():
    messages: List[Dict]

    def __init__(self, messages: List[Dict]):
        self.messages = messages

    def format(self, **kwargs):
        return [
            {
                "role":  m['role'],
                "content": m['content'].format(**kwargs)
            }
            for m in self.messages
        ]

In [ ]:
prompt_template = MessagesPromptTemplate(
    messages = [
        {"role": "system", "content": "You only answer in rhymes"},
        {"role": "user", "content": "Tell me about {city}"}
    ]
)

In [ ]:
prompt_template.format(city="Paris")

Давайте попробуем вызвать llm с разными переменными.

In [ ]:
outputs = nebius_client.chat.completions.create(
    messages=prompt_template.format(city="Paris"),
    model=llama_model
).choices[0].message.content
print(outputs)

In [ ]:
outputs = nebius_client.chat.completions.create(
    messages=prompt_template.format(city="Amsterdam"),
    model=llama_model,
).choices[0].message.content
print(outputs)

Написанный нами класс шаблона приглашения очень примитивен и завершится ошибкой, если, например, не будут введены некоторые клавиши.
Одну из хороших реализаций шаблонов промптов можно найти в LangChain [PromptTemplates](https://python.langchain.com/docs/concepts/prompt_templates/)

In [ ]:
!pip install langchain -qU

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "You only answer in rhymes"),
    ("user", "Tell me about {city}")
])

prompt_template.invoke({"city": "Madrid"})

**Примечание.** Вам не обязательно использовать вызовы LangChain llm или что-то еще, вы можете использовать только их реализацию PromptTemplate.
Однако в этой библиотеке есть немного полезного кода.

In [ ]:
from langchain_core.messages import convert_to_openai_messages

In [ ]:
templated_messages = convert_to_openai_messages(prompt_template.invoke({"city": "Madrid"}).to_messages())
templated_messages

In [ ]:
outputs = nebius_client.chat.completions.create(
    messages=templated_messages,
    model=llama_model,
).choices[0].message.content
print(outputs)

# Структурирование результатов LLM
Во многих случаях вам нужен не просто произвольный текстовый ответ, а что-то конкретное, что вы сможете использовать позже в своей системе. Например, если вы хотите, чтобы ваш LLM классифицировал намерение клиента, чтобы позже передать разговор соответствующему отделу, вам необходимо извлечь конкретный класс намерения из ответа LLM.
Чтобы удобно анализировать результаты LLM, разумно структурировать их определенным образом. Мы уже обсуждали некоторые приемы с промптами в Теме 1; на этот раз мы узнаем несколько более надежных способов заставить LLM соблюдать нестандартный формат вывода.

## Базовое структурирование вывода
В качестве основного способа структурирования результатов вы можете «попросить» LLM представить результаты в определенном формате. Например:

In [ ]:
outputs = nebius_client.chat.completions.create(
    messages=[{
        'role': 'user',
        'content': """Design one role play character\'s name, class and a short description.
Present it as a markdown list"""}],
    model=llama_model,
).choices[0].message.content
print(outputs)

Хотя это неплохо, но не очень надежно. Лучшим способом было бы показать LLM несколько примеров, чтобы он знал, чего мы ожидаем.
Эти примеры известны как **многократные примеры**, а сама техника промптов — как **многократные промпты**.

In [ ]:
outputs = nebius_client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': 'Design one role play character\'s name, class and a short description. Present it as a markdown list.\n'\
            "Examples:\n"\
            "\n"\
            "- **Name:** Randalf the Yellow;\n"\
            "- **Class:** Fire mage;\n"\
            "- **Proficiency:** Pyro magic;\n"\
            "- **Resistance:** Fire;\n"\
            "\n"\
            "- **Name:** Bonan;\n"\
            "- **Class:** Barbarian;\n"\
            "- **Proficiency:** Axe;\n"\
            "- **Resistance:** Mental magic;\n"\
        }
    ],
    model=llama_model,
).choices[0].message.content
print(outputs)

Как видите, LLM довольно хорошо уловил этот формат.

In [ ]:
outputs = nebius_client.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': 'Solve the following equation and output only the answer number without reasoning after "Answer:"\n' \
            '123 * 321 = ?\n' \
            'Answer:'
        }
    ],
    model=llama_model,
).choices[0].message.content
print(outputs)

Несмотря на то, что ответ неверен (LLM, как известно, плохо справляется с арифметикой), структура вывода правильна и ее легко разобрать.

Однако мы можем добиться еще большего.

## Структурированные результаты
Современные LLM поддерживают вывод в определенном формате, например, мы можем использовать «режим JSON», чтобы принудительно выводить выходные данные в формате JSON.

In [ ]:
json_output = nebius_client.chat.completions.create(
    messages=[{'role': 'user', 'content': 'Design a role play character\'s name, class and a short description in json format'}],
    model=llama_model,
    response_format={"type": "json_object"}
).choices[0].message.content
json_output

Это полезно, потому что впоследствии вам будет намного проще анализировать выходные данные:

In [ ]:
import json
json.loads(json_output)

Мы можем пойти еще дальше и фактически определить модель `pydantic` для создания схемы для наших выходных данных:

In [ ]:
from typing import List
from pydantic import BaseModel

class CharacterProfile(BaseModel):
    name: str
    age: int
    special_skills: List[str]
    traits: List[str]
    character_class: str
    origin: str

completion = nebius_client.chat.completions.create(
    model=llama_model,
    messages=[
        {"role": "user", "content": "Design a role play character"}
    ],
    extra_body={
        "guided_json": CharacterProfile.model_json_schema()
    }
)

CharacterProfile.model_validate_json(completion.choices[0].message.content)

Итак, нет, у нас есть предопределенный формат выходных данных, с которым легко работать.

Другой способ структурировать результаты — использовать примеры.
Давайте рассмотрим пример из известного [набора данных MMLU](https://huggingface.co/datasets/cais/mmlu):

In [ ]:
question = "Which of the following statements about Ethernets is typically FALSE?"

A = "Ethernets use circuit switching to send messages."
B = "Ethernets use buses with multiple masters."
C = "Ethernet protocols use a collision-detection method to ensure that messages are transmitted properly."
D = "Networks connected by Ethernets are limited in length to a few hundred meters."

correct_answer = "A"

В идеале мы хотим, чтобы наш LLM решил этот «тест», ответив нам письмом, соответствующим правильному ответу. Это также значительно облегчит расчет показателей. Посмотрим, что произойдет.

In [ ]:
output = nebius_client.chat.completions.create(
    messages=[{
        "role": "user",
        "content": f"""
Answer the following question with one of the options listed below
Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}
Answer:
"""}],
    model=llama_model,
).choices[0].message.content
print(output)

Как видите, он выдал правильный ответ, но если мы проведем простое сравнение, у нас возникнут проблемы:

In [ ]:
output == correct_answer

Итак, давайте научим нашу модель правильно отвечать, используя так называемую промпт с несколькими предложениями, также известную как контекстное обучение. По сути, мы показываем модели несколько примеров в промпте, чтобы научить ее, в каком формате мы хотим получить ответ.

In [ ]:
output = nebius_client.chat.completions.create(
    messages=[{
        "role": "user",
        "content": f"""
Examples:
Question: The IP protocol is primarily concerned with
A: Routing packets through the network
B: Reliable delivery of packets between directly connected machines
C: Reliable delivery of large (multi-packet) messages between machines that are not necessarily directly connected
D: Dealing with differences among operating system architectures
Answer:
A

Question: Which of the following is NOT a property of bitmap graphics?
A: Fast hardware exists to move blocks of pixels efficiently
B: Realistic lighting and shading can be done.
C: All line segments can be displayed as straight.
D: Polygons can be filled with solid colors and textures.
Answer:
A

Task:
Answer the following question with one of the options listed below. Only ouput the answer in the same format as the examples.
Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}
Answer:
"""}],
    model=llama_model,
).choices[0].message.content
print(output)

In [ ]:
output == correct_answer

Мы также заметили, что для некоторых моделей формат диалога на самом деле является лучшим способом структурирования примеров с несколькими кадрами.

In [ ]:
output = nebius_client.chat.completions.create(
    messages=[{
        "role": "user",
        "content": f"""
User: Answer the following question with one of the options listed below.
Question: The IP protocol is primarily concerned with
A: Routing packets through the network
B: Reliable delivery of packets between directly connected machines
C: Reliable delivery of large (multi-packet) messages between machines that are not necessarily directly connected
D: Dealing with differences among operating system architectures
Answer:
Assistant: A

User: Answer the following question with one of the options listed below.
Question: Which of the following is NOT a property of bitmap graphics?
A: Fast hardware exists to move blocks of pixels efficiently
B: Realistic lighting and shading can be done.
C: All line segments can be displayed as straight.
D: Polygons can be filled with solid colors and textures.
Answer:
Assistant: A

User: Answer the following question with one of the options listed below.
Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}
Answer:
Assitant:
"""}],
    model=llama_model,
).choices[0].message.content
print(output)

Теоретически нам даже не нужно показывать модели соответствующие примеры, если мы хотим, чтобы она изучила форматирование вывода.

In [ ]:
output = nebius_client.chat.completions.create(
    messages=[{
        "role": "user",
        "content": f"""
Question: Choose the letter A
A: A
B: B
C: C
D: D
Answer:
A

Question: Which is the biggest number?
A: 1
B: 2
C: 3
D: 4
Answer:
D

Answer the following question with one of the options listed below
Question: {question}
A: {A}
B: {B}
C: {C}
D: {D}
Answer:
"""}],
    model=llama_model,
).choices[0].message.content
print(output)

**Примечание.** Иногда вы можете спутать модель, если у вас есть примеры из распределения, которое отличается от вашего распределения. Поэтому для достижения наилучших результатов постарайтесь согласовать распределение.

## Вызов функции
Мы также можем использовать инструменты API OpenAI. Давайте посмотрим, как мы можем использовать веб-поиск, используя только API:

In [ ]:
!pip install tavily-python -qU

Нам понадобится ключ API Tavily, который вы можете получить [здесь](https://app.tavily.com/sign-in).
Затем либо используйте секретное хранилище Google, либо поместите его в файл и загрузите.

In [ ]:
#os.environ['TAVILITY_API_KEY"] = open(".tavily_api_key").read()
os.environ["TAVILY_API_KEY"] = userdata.get("tavily_api_key")

from tavily import TavilyClient

tavily_client = TavilyClient()

response = tavily_client.search("Who is Leo Messi?", topic="general")

print(response['results'])

Теперь мы можем определить описание `tool` для клиента, чтобы модель знала, как его использовать.
Мы будем предоставлять только параметры `query` и `topic`.
Нам также необходимо написать краткие описания, чтобы объяснить, для чего нужен инструмент и параметры. Обратите внимание: это не для вас, а для LLM :) Поэтому, пожалуйста, убедитесь, что вы предоставили четкое объяснение.
Использование инструмента является своего рода расширением «режима JSON», поскольку в конце мы получаем набор параметров, проанализированных из JSON.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "web-search",
            "description": "Retrieves results from web search",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "What you search for",
                    },
                    "topic": {
                        "type": "string",
                        "description": "Search topic either 'general' or 'news'",
                        "enum": ["general", "news"]
                    },
                },
                "required": ["query"],
            },
        }
    },
]


messages = []
messages.append({"role": "system", "content": "If you are asked about the factual information, create a function call instead. If you already searched, use the results to give an answer."})
messages.append({"role": "user", "content": "What is the name of the cat from Shrek?"})
chat_response = nebius_client.chat.completions.create(
    messages=messages, tools=tools, model=llama_model
)
chat_response

И мы также можем попытаться запросить какой-нибудь достойный новостей контент, чтобы узнать, примет ли LLM другое решение `topic`.

In [ ]:
messages = []
messages.append({"role": "system", "content": "If you are asked about the factual information, create a function call instead. If you already searched, use the results to give an answer."})
messages.append({"role": "user", "content": "What happened in London today?"})
chat_response = nebius_client.chat.completions.create(
    messages=messages, tools=tools, model=llama_model
)
chat_response

Теперь мы можем извлечь вывод использования функции из результата.

In [ ]:
chat_response.choices[0].message.tool_calls[0]

Вам может быть интересно, почему мы включаем использование инструментов в тему структурированного вывода.
Дело в том, что вы также можете использовать эту функцию для структурирования вывода. Вам не обязательно использовать реальную функцию в качестве инструмента. Давайте воспользуемся нашим предыдущим примером

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "create_rpg_character",
            "description": "Creates a character based on attributes and description",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Name of the character",
                    },
                    "age": {
                        "type": "integer",
                        "description": "Age of the character",
                    },
                    "special_skills": {
                        "type": "array",
                        "description": "List of special skills of the character",
                        "items": {
                            "type": "string"
                        }
                    },
                    "traits": {
                        "type": "array",
                        "description": "List of traits of the character",
                        "items": {
                            "type": "string"
                        }
                    },
                    "character_class": {
                        "type": "string",
                        "description": "Class of the character",
                        "enum": ["mage", "rogue", "barbarian", "knight", "paladin"]
                    },
                    "origin": {
                        "type": "string",
                        "description": "Origin of the character",
                        "enum": ["human", "elf", "orc", "undead"]
                    },
                },
                "required": ["name", "age", "special_skills", "traits", "character_class", "origin"],
            },
        }
    },
]


In [ ]:
messages = []
messages.append({"role": "system", "content": "If you are asked to create a character, use `create_rpg_character` tool."})
messages.append({"role": "user", "content": "Generate a random character for my new session"})
chat_response = nebius_client.chat.completions.create(
    messages=messages, tools=tools, model=llama_model
)
chat_response.choices[0].message.tool_calls[0].function.arguments

# **Практические задания**
Если вы столкнулись с какими-либо трудностями или просто хотите увидеть наши решения, загляните в [Блокнот решений](https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/2.1_structured_inputs_and_outputs_solutions.ipynb).

## Задача 1. LLM Извлечение информации
Целью данной задачи является создание системы, которая извлекает данные о событиях из свободного текста в предсказуемый формат.

Представим, что вы работаете в маркетинговом агентстве, и вам нужно собрать аналитику о проходящих мероприятиях, посвященных AI и Machine Learning. Для этого вам необходимо обработать пресс-релизы и извлечь:
- Название мероприятия,
- Дата мероприятия,
- Количество участников,
- Количество динамиков,
- Стоимость участия.
Конечно, вы можете сделать это вручную, но гораздо интереснее использовать генеративный ИИ! Итак, вашей задачей будет написать функцию, которая сделает это всего одним запросом к OpenAI API.
Ниже приведен пример пресс-релиза (конечно, сгенерированного ChatGPT, так что и событие, и персонажи являются вымышленными). Все они находятся в архиве press_releases.zip в папке домашнего задания 1-й недели.
<блок-цитата>
<p>ПРЕСС-РЕЛИЗ
InnovAI Summit 2023: Взгляд в будущее искусственного интеллекта</p>
Город добродетели, Киберпространство – 8 ноября 2023 г. – Самое ожидаемое событие года, InnovAI Summit 2023, успешно завершилось в минувшие выходные, 5 ноября 2023 г. Саммит, проходивший на современной арене VirtuTech Arena, собрал более 3500 участников, от блестящих энтузиастов и исследователей в области искусственного интеллекта до пионеров в этой области.
Уважаемые докладчики вышли на сцену, чтобы пролить свет на последние достижения, практические реализации и этические соображения в области искусственного интеллекта. Доктор Эвелин Квантум, известная своей новаторской работой в области квантового машинного обучения, подчеркнула важность этого слияния и то, как оно произведет революцию в компьютерных технологиях, какими мы их знаем. Еще один основной доклад прозвучал от профессора Лео Нексуса, чей текущий проект «ИИ для устойчивого развития» подчеркивает симбиотические отношения между природой и машиной, стремясь использовать ИИ для восстановления экосистем нашей планеты.
В этом году панельная дискуссия, модератором которой выступила талантливый доктор Ада Нейра, включала оживленные дебаты о пределах возможностей ИИ в творческом искусстве. Известный цифровой художник Феликс Вортекс продемонстрировал, как он использует генеративно-состязательные сети для создания сюрреалистических произведений искусства, а автор бестселлеров Айрис Лум рассказала о своих экспериментах с созданием историй с помощью искусственного интеллекта.
Среди других ярких моментов были практические семинары, интерактивные сессии вопросов и ответов, а также дебаты на тему «ИИ и этика», которые были особенно хорошо приняты и подчеркивали необходимость прозрачности и справедливости в моделях ИИ. Эксклюзивная «Аллея стартапов» позволила начинающим предпринимателям продемонстрировать свои инновации, привлекая внимание мировых венчурных капиталистов и средств массовой информации.
Мероприятие завершилось объявлением о проведении InnovAI Summit 2024, который обещает быть еще более грандиозным. Участники ушли с новым энтузиазмом по поводу огромных возможностей, которые обещает мир искусственного интеллекта и машинного обучения.
По вопросам СМИ обращайтесь:
Джейн Сайфер
Директор по коммуникациям InnovAI Summit
Электронная почта: jane.cipher@innovai.org
Телефон: +123-4567-8910</p>
</blockquote>
Точнее, вам следует написать функциюион
```python
parse_press_release(pr: str) -> dict
```
где вывод должен быть в формате
```python
{
  name: 'InnovAI Summit 2023',
  date: '08.11.2023',
  n_participants: 3500,
  n_speakers: 4,
  price:
}
```
Если какая-либо из четырех характеристик не упомянута в тексте, поставьте `None` в соответствующее поле.
В конце подсчитайте статистику правильных ответов и проанализируйте, какие ошибки вы «моделируете» допускаете больше всего.

**Советы и предложения:**
- Экспериментировать на игровой площадке Nebius AI Studio будет удобнее https://studio.nebius.com/playground.
- Вам нужно очень точно определить, чего вы хотите от модели.
- Будет полезно, если вы укажете в приглашении, что вывод должен быть в формате JSON, так вы потратите меньше времени на анализ вывода. Но будьте осторожны. Хотя некоторые модели легко выводят запрос на вывод JSON, проверьте формат вывода. Он может содержать излишнее форматирование, например:
<pre><code>```json
{"name": "InnovAI Summit 2023", ...}
```</pre></code>
На самом деле, изучение результатов LLM и их формата является обязательным при работе с ними.
- Пожалуйста, будьте осторожны с деталями. Например, Джейн Сайфер в приведенном выше тексте не является оратором и не должна быть контратакой как таковой (как избавиться от контактного лица?). Также обратите внимание на формат даты,
- Если модель слишком своенравна с выходным форматом, не стесняйтесь показать несколько примеров. Снижение температуры прогнозов может помочь снизить креативность ответа, а это то, что нам нужно для такой задачи.
- Отладка приложения на базе LLM может оказаться непростой задачей. Когда вы думаете, что уже отточили это, степень магистра все еще может вас удивить. Поэтому мы не ожидаем 100% точности в этой задаче, но ожидаем, что вы приложите все усилия для достижения качественного результата.

**Бонусные баллы**:
Попробуйте написать решение, используя:
- Структурированный вывод JSON
- Управление выводом JSON с помощью структур.

In [ ]:
press_release = """PRESS RELEASE

InnovAI Summit 2023: A Glimpse into the Future of Artificial Intelligence

City of Virtue, Cyberspace - November 8, 2023 - The most anticipated event of the year, InnovAI Summit 2023, successfully concluded last weekend, on November 5, 2023. Held in the state-of-the-art VirtuTech Arena, the summit saw a massive turnout of over 3,500 participants, from brilliant AI enthusiasts and researchers to pioneers in the field.

Esteemed speakers took to the stage to shed light on the latest breakthroughs, practical implementations, and ethical considerations in AI. Dr. Evelyn Quantum, renowned for her groundbreaking work on Quantum Machine Learning, emphasized the importance of this merger and how it's revolutionizing computing as we know it. Another keynote came from Prof. Leo Nexus, whose current project 'AI for Sustainability' highlights the symbiotic relationship between nature and machine, aiming to use AI in restoring our planet's ecosystems.

This year's panel discussion, moderated by the talented Dr. Ada Neura, featured lively debates on the limits of AI in creative arts. Renowned digital artist, Felix Vortex, showcased how he uses generative adversarial networks to create surreal art pieces, while bestselling author, Iris Loom, explained her experiments with AI-assisted story crafting.

Among other highlights were hands-on workshops, interactive Q&A sessions, and an 'AI & Ethics' debate which was particularly well-received, emphasizing the need for transparency and fairness in AI models. An exclusive 'Start-up Alley' allowed budding entrepreneurs to showcase their innovations, gaining attention from global venture capitalists and media.

The event wrapped up with an announcement for InnovAI Summit 2024, set to be even grander. Participants left with a renewed enthusiasm for the vast possibilities that the AI and ML world promises.

For media inquiries, please contact: Jane Cipher Director of Communications, InnovAI Summit Email: jane.cipher@innovai.org Phone: +123-4567-8910"""

In [ ]:
def parse_press_release(pr: str) -> dict:
    pass

### Тестирование
Мы подготовили небольшой набор данных, чтобы вы могли проверить свою промпт. Если вы написали свою функцию, попробуйте запустить следующий код. В конце у вас также есть возможность просмотреть результаты в таблице рядом с файлом with_results.csv. Ваша цель — правильно заполнить хотя бы 60 % полей.

In [ ]:
!pip install --upgrade gdown
!gdown -O press_release_extraction.csv https://docs.google.com/spreadsheets/d/15IGdc3MV8864lxrLxsug0Ij480p76T1EAwBM7WGT_OI/export?format=csv

In [ ]:
import pandas
pr_df = pandas.read_csv("press_release_extraction.csv")
pr_df.head()

In [ ]:
pr_df.pr_parsed[0]

In [ ]:
import json

parsed_list = []
fields = {
    "name": str,
    "date": str,
    "n_speakers": int,
    "n_participants": int,
    "price": str
}
correct_fields = 0
for row in pr_df.itertuples():
    parsed_release = parse_press_release(row.pr_text)
    parsed_list.append(json.dumps(parsed_release, indent=4))
    golden = json.loads(row.pr_parsed)
    for field, field_type in fields.items():
        golden_field = golden[field]
        parsed_field = parsed_release.get(field)
        try:
            parsed_field = field_type(parsed_field)
        except (ValueError, TypeError):
            pass
        if golden_field == parsed_field:
            correct_fields += 1
        else:
            print(f"For {golden['name']} {field} {parsed_release.get(field)} doesn't seem the same as {golden[field]}")

print(f"Correctly extracted {correct_fields} out of {5*len(pr_df)}")

### Бонусные баллы
- Попробуйте и сравните разные способы установления правильного формата ответа.
- Попробуйте и сравните разные LLM

## Задача 2. Локализованный MMLU
Самое интересное в структурированном выводе то, что очень легко сделать переведенную версию определенного набора данных, принимая во внимание весь контекст и выводя в формате, который очень легко анализировать. Давайте попробуем это на MMLU.
**Задача:** Напишите функцию, которая вводит образец из MMLU и выводит переведенную версию, используя структурированные выходные данные.
Совет: следите за тем, чтобы правильный ответ не изменился.

In [ ]:
!pip install -qU datasets

In [ ]:
from typing import List
from pydantic import BaseModel

class MMLUSample(BaseModel):
    ...

def translate_mmlu_sample(sample: MMLUSample, target_language: str) -> MMLUSample:
    ...

In [ ]:
from typing import List
from pydantic import BaseModel

class MMLUSample(BaseModel):
    question: str
    A: str
    B: str
    C: str
    D: str
    correct_answer: str

def translate_mmlu_sample(sample: MMLUSample, target_language: str) -> MMLUSample:
    completion = nebius_client.chat.completions.create(
        model=llama_model,
        messages=[
            {
                "role": "user",
                "content": f"Translate this MMLU sample into {target_language}" \
                f"Question: {sample.question}\n" \
                f"A: {sample.A}\n" \
                f"B: {sample.B}\n" \
                f"C: {sample.C}\n" \
                f"D: {sample.D}\n" \
                f"Correct answer: {sample.correct_answer}\n" \
                f"Translated sample:"
            }
        ],
        extra_body={
            "guided_json": MMLUSample.model_json_schema()
        },
    )

    translated = MMLUSample.model_validate_json(completion.choices[0].message.content)
    if translated.correct_answer != sample.correct_answer:
        translated.correct_answer = sample.correct_answer
    return translated

In [ ]:
mmlu_sample = MMLUSample(
    question = "Which of the following statements about Ethernets is typically FALSE?",
    A = "Ethernets use circuit switching to send messages.",
    B = "Ethernets use buses with multiple masters.",
    C = "Ethernet protocols use a collision-detection method to ensure that messages are transmitted properly.",
    D = "Networks connected by Ethernets are limited in length to a few hundred meters.",
    correct_answer = "A"
)

translate_mmlu_sample(mmlu_sample, target_language="German")

Теперь давайте вспомним код, который мы написали для оценщика MMLU, и добавим небольшую особенность:
У нас будет и тема, и язык, на котором мы хотим оценить модель.

In [ ]:
!pip install datasets -q

**Задача**: Измените следующий код MMLUEvaluator, чтобы он также мог переводить входной вопрос и оценивать производительность на другом языке.

In [ ]:
import pandas as pd
from typing import List, Dict, Tuple
import json
from pathlib import Path
import numpy as np
from tqdm import tqdm

from datasets import load_dataset

class MMLUEvaluator:
    def __init__(self, system_prompt: str = None, prompt: str = None,
                 topic: str = "high_school_mathematics"):
        """
        Initialize the MMLU evaluator.

        Args:
            system_prompt: Optional system prompt for the model
            prompt: Custom prompt for the model
            topic: Which topic to choose
        """

        self.topic = topic
        self.topic_prettified = topic.replace("_", " ")
        self.system_prompt = system_prompt or f"You are an expert in {self.topic_prettified}."

        self.prompt = """You are given a question in {topic_prettified} with four answer options labeled by A, B, C, and D.
You need to ponder the question and justify the choice of one of the options A, B, C, or D.
At the end, do write the chosen answer option A, B, C, D after #ANSWER:
Now, take a deep breath and work out this problem step by step. If you do well, I'll tip you 200$.

QUESTION: {question}

ANSWER OPTIONS:
A: {A}
B: {B}
C: {C}
D: {D}
"""

        self.questions, self.choices, self.answers = self.load_mmlu_data(topic=self.topic)

    def load_mmlu_data(self, topic: str) -> pd.DataFrame:
        """
        Load MMLU test data on a given topic.

        Args:
            topic: Which topic to choose

        Returns:
            DataFrame with questions and answers
        """

        dataset = load_dataset("cais/mmlu", topic, split="test")

        dataset = dataset
        dataset = pd.DataFrame(dataset)

        # Load questions and choices separately
        questions = dataset["question"]
        choices = pd.DataFrame(
            data=dataset["choices"].tolist(), columns=["A", "B", "C", "D"]
        )
        # In the dataset, true answer labels are in 0-3 format;
        # We convert it to A-D
        answers = dataset["answer"].map(lambda ans: {0: "A", 1: "B", 2: "C", 3: "D"}[ans])

        return questions, choices, answers

    def extract_answer(self, solution: str) -> str:
        """
        Extract the letter answer from model's response.

        Args:
            response: Raw model response

        Returns:
            Extracted answer letter (A, B, C, D, or Failed to parse)
        """
        # Look for a single letter answer in the response
        try:
            answer = solution.split('#ANSWER:')[1].strip()
        except:
            answer = "Failed to parse"
        return answer

    def evaluate_single_question(self, question: str, choices: Dict[str, str],
                                 correct_answer: str,
                                 client, model) -> Tuple[bool, str]:
        """
        Evaluate a single question.

        Args:
            question: Formatted question string
            correct_answer: Correct answer letter

        Returns:
            Tuple of (is_correct, extracted_answer, model_response)
        """
        try:
            model_response = answer_with_llm(
                prompt=self.prompt.format(
                    client=client, model=model,
                    topic_prettified=self.topic_prettified,
                    question=question,
                    A=choices['A'], B=choices['B'], C=choices['C'], D=choices['D']
                ),
                system_prompt=self.system_prompt,
                prettify=False
            )
            answer = self.extract_answer(model_response)
            is_correct = (answer.upper() == correct_answer.upper())
            return is_correct, answer, model_response
        except Exception as e:
            print(f"Error evaluating question: {e}")
            return False, None, None

    def run_evaluation(self, client=nebius_client, model="meta-llama/Meta-Llama-3.1-8B-Instruct",
                       n_questions=50) -> Dict:
        """
        Run evaluation of a given model on the first n_questions.

        Args:
            client: Which client to use (OpenAI or Nebius)
            model: Which model to use
            n_questions: How many first questions to take

        Returns:
            Dictionary with evaluation metrics
        """
        evaluation_log = []
        correct_count = 0

        if n_questions:
            n_questions = min(n_questions, len(self.questions))
        else:
            n_questions = len(self.questions)

        for i in tqdm(range(n_questions)):
            is_correct, answer, model_response = self.evaluate_single_question(
                question=self.questions[i],
                choices=self.choices.iloc[i],
                correct_answer=self.answers[i],
                client=client,
                model=model,
            )

            if is_correct:
                correct_count += 1

            evaluation_log.append({
                'answer': answer,
                'model_response': model_response,
                'is_correct': is_correct
            })

        accuracy = correct_count / n_questions
        evaluation_results = {
            'accuracy': accuracy,
            'evaluation_log': evaluation_log
        }

        return evaluation_results


### Тестирование

In [ ]:
evaluator = MMLUEvaluator(topic="medical_genetics", language="English")

results = evaluator.run_evaluation(model="meta-llama/Meta-Llama-3.1-8B-Instruct",
                         n_questions=50)
print(f'\nAccuracy: {results["accuracy"]}')

In [ ]:
evaluator_de = MMLUEvaluator(topic="medical_genetics", language="German")

results_de = evaluator_de.run_evaluation(model="meta-llama/Meta-Llama-3.1-8B-Instruct",
                         n_questions=10)
print(f'\nAccuracy: {results_de["accuracy"]}')